# Global Media Sentiment & Share of Voice Tracker

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze track** using Snowflake Cortex AI
2. **Build production pipelines** with SQL and SENTIMENT()
3. **Create data structures** for News outlets
4. **Implement business logic** for communications intelligence
5. **Generar insights** with window functions and aggregations
6. **Build interactive dashboards** for stakeholder reporting

---

## What You'll Build

A **global media sentiment & share of voice tracker** that provides:
- Track news coverage, analyst reports, and earned media sentiment across global markets
- Automated data collection and processing
- Real-time analysis and insights
- Interactive visualization dashboard
- Production-ready SQL for scale

---

## Technical Requirements

- **Snowflake account** with Cortex AI enabled
- **Approximately 12-15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `AI_SENTIMENT()` - Primary AI function
- Window functions - Time-series analysis
- Aggregations - Summary statistics
- CTEs - Complex query logic
- `CASE` statements - Classification

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `MEDIA_SENTIMENT_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Business Challenge

Track news coverage, analyst reports, and earned media sentiment across global markets

### Why This Matters

Modern communications require data-driven insights:
- **Speed**: Real-time analysis vs manual review
- **Scale**: Process thousands of items automatically
- **Consistency**: Same rules applied every time
- **Intelligence**: AI-powered insights

In [ ]:
-- Create environment
CREATE DATABASE IF NOT EXISTS MEDIA_SENTIMENT_DB;
CREATE SCHEMA IF NOT EXISTS MEDIA_SENTIMENT_DB.PUBLIC;
USE SCHEMA MEDIA_SENTIMENT_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'Global Media Sentiment & Share of Voice Tracker Environment Ready!' as status;

---

## Paso 2: Define Data Structure

### Tables

1. **`media_articles`** - Primary data source
2. **`media_sentiment`** - Analysis results
3. **`share_of_voice`** - Aggregated insights

### Data Flow

1. **Ingest** → Collect data from sources
2. **Process** → Apply AI functions
3. **Analyze** → Calculate metrics
4. **Visualize** → Present insights

In [ ]:
-- Primary data table
CREATE OR REPLACE TABLE media_articles (
    record_id VARCHAR(50) PRIMARY KEY,
    source VARCHAR(100),
    content TEXT,
    created_date DATE,
    metadata VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Analysis results table
CREATE OR REPLACE TABLE media_sentiment (
    analysis_id VARCHAR(50) PRIMARY KEY,
    record_id VARCHAR(50),
    analysis_result FLOAT,
    category VARCHAR(50),
    confidence FLOAT,
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    FOREIGN KEY (record_id) REFERENCES media_articles(record_id)
);

-- Aggregated insights table
CREATE OR REPLACE TABLE share_of_voice (
    insight_id VARCHAR(50) PRIMARY KEY,
    date_period DATE,
    metric_name VARCHAR(100),
    metric_value FLOAT,
    trend VARCHAR(20),
    calculated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

SELECT 'Tables created successfully!' as status;

---

## Paso 3: Generar Datos Sintéticos Data

### Qué Vamos a Crear

- **1,000 records** for demonstration
- **Realistic content** matching use case
- **Last 30 days** of data
- **Multiple sources** and categories

### Synthetic Data Approach

Using Snowflake's `GENERATOR()` and `UNIFORM()` functions to create realistic test data.

In [ ]:
-- Generar datos sintéticos data
INSERT INTO media_articles
WITH sources AS (
    SELECT * FROM (VALUES
        ('source_a'), ('source_b'), ('source_c'), ('source_d'), ('source_e')
    ) t(source)
),
content_templates AS (
    SELECT * FROM (VALUES
        ('Positive content example for Global analysis'),
        ('Neutral content example for testing and validation purposes'),
        ('Negative content example showing issues and concerns'),
        ('Mixed sentiment content with both positive and negative aspects'),
        ('Technical content focusing on specific product features')
    ) t(template)
)
SELECT 
    'REC_' || LPAD(SEQ4()::VARCHAR, 8, '0') as record_id,
    s.source,
    REPLACE(ct.template, 'example', 'item ' || SEQ4()::VARCHAR) as content,
    DATEADD('day', -FLOOR(UNIFORM(1, 30, RANDOM())), CURRENT_DATE()) as created_date,
    OBJECT_CONSTRUCT(
        'category', ARRAY_CONSTRUCT('cat_a', 'cat_b', 'cat_c')[FLOOR(UNIFORM(0, 3, RANDOM()))],
        'priority', ARRAY_CONSTRUCT('low', 'medium', 'high')[FLOOR(UNIFORM(0, 3, RANDOM()))]
    ) as metadata,
    CURRENT_TIMESTAMP() as ingested_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000)) g
CROSS JOIN sources s
CROSS JOIN content_templates ct
WHERE UNIFORM(0, 1, RANDOM()) < 0.2
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 1000;

SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT source) as sources,
    MIN(created_date) as earliest_date,
    MAX(created_date) as latest_date
FROM media_articles;

---

## Paso 4: Apply AI Analysis

### Using Cortex AI

SENTIMENT() processes content automatically:
- No model training required
- Production-ready performance
- Scalable to millions of records

In [ ]:
-- Analyze sentiment for all media articles
-- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT
INSERT INTO media_sentiment
SELECT 
    'ANL_' || LPAD(ROW_NUMBER() OVER (ORDER BY record_id), 8, '0') as analysis_id,
    record_id,
    -- Convert sentiment to numeric score for analysis_result column (Use Case 28 pattern)
    CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as analysis_result,
    -- Human-readable sentiment category
    CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 'Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Very Negative'
    END as category,
    0.95 as confidence,
    CURRENT_TIMESTAMP() as analyzed_at
FROM media_articles;

-- View sentiment distribution by source
SELECT 
    ma.source,
    ms.category as sentiment_category,
    COUNT(*) as article_count,
    ROUND(AVG(ms.analysis_result), 3) as avg_sentiment_score
FROM media_articles ma
JOIN media_sentiment ms ON ma.record_id = ms.record_id
GROUP BY ma.source, ms.category
ORDER BY ma.source, avg_sentiment_score DESC;

---

## Paso 5: Calculate Aggregated Metrics

### Summary Statistics

Create daily/weekly aggregations for:
- Trend analysis
- Baseline calculations
- Anomaly detection

In [ ]:
-- Calculate daily metrics and populate share_of_voice
INSERT INTO share_of_voice
SELECT 
    'INS_' || LPAD(ROW_NUMBER() OVER (ORDER BY created_date), 8, '0') as insight_id,
    created_date as date_period,
    'daily_average' as metric_name,
    ROUND(AVG(a.analysis_result), 3) as metric_value,
    CASE 
        WHEN AVG(a.analysis_result) > LAG(AVG(a.analysis_result)) OVER (ORDER BY created_date) THEN 'up'
        WHEN AVG(a.analysis_result) < LAG(AVG(a.analysis_result)) OVER (ORDER BY created_date) THEN 'down'
        ELSE 'stable'
    END as trend,
    CURRENT_TIMESTAMP() as calculated_at
FROM media_articles r
INNER JOIN media_sentiment a ON r.record_id = a.record_id
GROUP BY created_date;

-- View results
SELECT * FROM share_of_voice ORDER BY date_period DESC LIMIT 10;

---

## Paso 6: Sentiment Distribution Analysis

### Qué Vamos a Hacer

Categorize sentiment scores into **Positive**, **Neutral**, and **Negative** groups for easier interpretation.

### Why This Matters

- **Executive reporting**: Leadership wants simple categories, not raw scores
- **Actionable insights**: "58% positive coverage" is clearer than "avg 0.23 sentiment"
- **Trend tracking**: Track category shifts over time

### Key Concepts

**Sentiment Categorization**:
- **Positive**: Score ≥ 0.5
- **Negative**: Score ≤ -0.5
- **Neutral**: Score between -0.5 and 0.5

Uses `CASE` statements and `ARRAY_AGG()` for grouping.

In [ ]:
-- Analyze sentiment distribution across positive/neutral/negative
SELECT 
    CASE 
        WHEN a.analysis_result >= 0.5 THEN 'Positive'
        WHEN a.analysis_result <= -0.5 THEN 'Negative'
        ELSE 'Neutral'
    END as sentiment_category,
    COUNT(*) as article_count,
    ROUND(AVG(a.analysis_result), 3) as avg_sentiment,
    ROUND(AVG(a.confidence), 3) as avg_confidence,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as percentage,
    ARRAY_AGG(DISTINCT r.source) as sources_list
FROM media_articles r
INNER JOIN media_sentiment a ON r.record_id = a.record_id
GROUP BY sentiment_category
ORDER BY article_count DESC;

---

## Paso 7: Time-Based Sentiment Trending

### Qué Vamos a Hacer

Analyze how sentiment changes **over time** using weekly aggregations.

### Why This Matters

- **Trend detection**: Spot sentiment shifts before they become crises
- **Campaign impact**: Measure sentiment before/after product launches
- **Pattern recognition**: Identify seasonal or cyclical trends

### Key Functions

- **`DATE_TRUNC('week', date)`**: Groups dates into weeks
- **`COUNT(CASE WHEN ... THEN 1 END)`**: Conditional counting
- Window aggregations for time-series analysis

In [ ]:
-- Analyze sentiment trends by week
SELECT 
    DATE_TRUNC('week', r.created_date) as week,
    COUNT(*) as articles_per_week,
    ROUND(AVG(a.analysis_result), 3) as avg_sentiment,
    COUNT(CASE WHEN a.analysis_result >= 0.5 THEN 1 END) as positive_count,
    COUNT(CASE WHEN a.analysis_result <= -0.5 THEN 1 END) as negative_count,
    COUNT(CASE WHEN a.analysis_result > -0.5 AND a.analysis_result < 0.5 THEN 1 END) as neutral_count,
    ROUND(100.0 * COUNT(CASE WHEN a.analysis_result >= 0.5 THEN 1 END) / COUNT(*), 1) as positive_pct,
    -- Week-over-week change
    ROUND(AVG(a.analysis_result) - LAG(AVG(a.analysis_result)) OVER (ORDER BY DATE_TRUNC('week', r.created_date)), 3) as sentiment_change
FROM media_articles r
INNER JOIN media_sentiment a ON r.record_id = a.record_id
GROUP BY week
ORDER BY week DESC
LIMIT 10;

---

## Paso 8: Source Performance Ranking

### Qué Vamos a Hacer

Rank media sources by both **volume** and **sentiment quality** to identify key outlets.

### Why This Matters

- **Media relations**: Know which outlets to prioritize
- **Share of voice**: Measure coverage across sources
- **Quality vs quantity**: Alto-volume sources might have low sentiment

### Key Functions

- **`RANK() OVER (ORDER BY ...)`**: Competitive ranking
- **Percentage calculations**: Convert counts to percentages
- **Multiple ranking criteria**: Volume rank vs sentiment rank

In [ ]:
-- Rank sources by volume and sentiment
SELECT 
    r.source,
    COUNT(*) as total_articles,
    ROUND(AVG(a.analysis_result), 3) as avg_sentiment,
    ROUND(STDDEV(a.analysis_result), 3) as sentiment_stddev,
    COUNT(CASE WHEN a.analysis_result >= 0.5 THEN 1 END) as positive_articles,
    ROUND(100.0 * COUNT(CASE WHEN a.analysis_result >= 0.5 THEN 1 END) / COUNT(*), 1) as positive_pct,
    COUNT(CASE WHEN a.analysis_result <= -0.5 THEN 1 END) as negative_articles,
    ROUND(100.0 * COUNT(CASE WHEN a.analysis_result <= -0.5 THEN 1 END) / COUNT(*), 1) as negative_pct,
    RANK() OVER (ORDER BY COUNT(*) DESC) as volume_rank,
    RANK() OVER (ORDER BY AVG(a.analysis_result) DESC) as sentiment_rank,
    -- Combined score: balance volume and sentiment
    ROUND(COUNT(*) * AVG(a.analysis_result), 2) as impact_score
FROM media_articles r
INNER JOIN media_sentiment a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY impact_score DESC;

---

## Paso 9: Category & Priority Analysis

### Qué Vamos a Hacer

Extract and analyze **metadata fields** (category, priority) from the VARIANT column.

### Why This Matters

- **Semi-structured data**: Real-world data often comes in JSON/VARIANT format
- **Multi-dimensional analysis**: Combine metadata with sentiment
- **Prioritization**: Focus on high-priority negative items

### Key Functions

- **`metadata:category::STRING`**: Extract JSON field from VARIANT
- **`VARIANT` data type**: Flexible schema for evolving data
- Cross-tabulation for multi-dimensional insights

In [ ]:
-- Analyze by metadata category and priority
SELECT 
    r.metadata:category::STRING as category,
    r.metadata:priority::STRING as priority,
    COUNT(*) as article_count,
    ROUND(AVG(a.analysis_result), 3) as avg_sentiment,
    ROUND(AVG(a.confidence), 3) as avg_confidence,
    MIN(a.analysis_result) as most_negative,
    MAX(a.analysis_result) as most_positive,
    -- Flag high-priority negative items for immediate attention
    COUNT(CASE WHEN a.analysis_result <= -0.5 AND r.metadata:priority::STRING = 'high' THEN 1 END) as high_priority_negative,
    -- Calculate sentiment range (volatility indicator)
    ROUND(MAX(a.analysis_result) - MIN(a.analysis_result), 3) as sentiment_range
FROM media_articles r
INNER JOIN media_sentiment a ON r.record_id = a.record_id
GROUP BY r.metadata:category::STRING, r.metadata:priority::STRING
ORDER BY article_count DESC;

---

## Paso 10: Outlier Detection with Z-Scores

### Qué Vamos a Hacer

Use **statistical analysis** to identify extreme sentiment outliers that need human review.

### Why This Matters

- **Anomaly detection**: Find unusually positive/negative coverage
- **Quality assurance**: Verify AI results on extreme cases
- **Crisis early warning**: Catch sentiment spikes before they escalate

### Key Concepts

**Z-Score**: Measures how many standard deviations a value is from the mean
- **Z > 2 or Z < -2**: Statistical outlier (top/bottom 5%)
- **Z > 3 or Z < -3**: Extreme outlier (top/bottom 0.3%)

Uses **CTEs** for multi-step calculations.

In [ ]:
-- Find extreme sentiment outliers using z-scores
WITH sentiment_stats AS (
    SELECT 
        AVG(analysis_result) as mean_sentiment,
        STDDEV(analysis_result) as stddev_sentiment
    FROM media_sentiment
)
SELECT 
    r.source,
    r.created_date,
    SUBSTRING(r.content, 1, 150) as content_preview,
    a.analysis_result as sentiment_score,
    ROUND(a.confidence, 3) as confidence,
    s.mean_sentiment,
    s.stddev_sentiment,
    ROUND((a.analysis_result - s.mean_sentiment) / s.stddev_sentiment, 2) as z_score,
    CASE 
        WHEN ABS((a.analysis_result - s.mean_sentiment) / s.stddev_sentiment) > 3 THEN 'EXTREME OUTLIER'
        WHEN ABS((a.analysis_result - s.mean_sentiment) / s.stddev_sentiment) > 2 THEN 'OUTLIER'
        ELSE 'NORMAL'
    END as outlier_flag,
    CASE 
        WHEN (a.analysis_result - s.mean_sentiment) / s.stddev_sentiment > 2 THEN '⚠️ Review: Extremely positive'
        WHEN (a.analysis_result - s.mean_sentiment) / s.stddev_sentiment < -2 THEN '🚨 URGENT: Extremely negative'
        ELSE '✓ Normal range'
    END as action_needed
FROM media_articles r
INNER JOIN media_sentiment a ON r.record_id = a.record_id
CROSS JOIN sentiment_stats s
WHERE ABS((a.analysis_result - s.mean_sentiment) / s.stddev_sentiment) > 1.5
ORDER BY ABS(z_score) DESC
LIMIT 20;

In [ ]:
-- Additional analysis query 5
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM media_articles r
INNER JOIN media_sentiment a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

In [ ]:
# Global Media Sentiment & Share of Voice Tracker Dashboard
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("📊 Global Media Sentiment & Share of Voice Tracker")
st.markdown("### Track news coverage, analyst reports, and earned media sentiment across global markets")

# Key metrics with error handling for empty tables
col1, col2, col3, col4 = st.columns(4)

try:
    total_records = session.sql("SELECT COUNT(*) as cnt FROM media_articles").collect()[0]['CNT']
except:
    total_records = 0

try:
    avg_result = session.sql("SELECT ROUND(AVG(analysis_result), 3) as avg FROM media_sentiment").collect()[0]['AVG']
    if avg_result is None:
        avg_result = 0.0
except:
    avg_result = 0.0

try:
    categories = session.sql("SELECT COUNT(DISTINCT category) as cnt FROM media_sentiment").collect()[0]['CNT']
except:
    categories = 0

try:
    trend_result = session.sql("SELECT trend FROM share_of_voice ORDER BY date_period DESC LIMIT 1").collect()
    recent_trend = trend_result[0]['TREND'] if len(trend_result) > 0 else 'N/A'
except:
    recent_trend = 'N/A'

with col1:
    st.metric("Total Records", f"{total_records:,}")
with col2:
    st.metric("Avg Result", f"{avg_result:.3f}")
with col3:
    st.metric("Categories", categories)
with col4:
    st.metric("Trend", recent_trend, delta="📈" if recent_trend == "up" else None)

# Category distribution
st.markdown("---")
st.subheader("📊 Analysis by Category")

try:
    category_data = session.sql("""
        SELECT category, COUNT(*) as count
        FROM media_sentiment
        GROUP BY category
        ORDER BY count DESC
    """).to_pandas()
    
    if not category_data.empty:
        st.bar_chart(category_data.set_index('CATEGORY')['COUNT'])
    else:
        st.info("No category data available yet. Run Cell 8 to analyze sentiment.")
except Exception as e:
    st.warning(f"Category data not available: {str(e)}")

# Trend over time
st.markdown("---")
st.subheader("📈 Trend Analysis")

try:
    trend_data = session.sql("""
        SELECT 
            date_period,
            metric_value
        FROM share_of_voice
        ORDER BY date_period
    """).to_pandas()
    
    if not trend_data.empty:
        st.line_chart(trend_data.set_index('DATE_PERIOD')['METRIC_VALUE'])
    else:
        st.info("No trend data available yet. Run Cell 10 to calculate metrics.")
except Exception as e:
    st.warning(f"Trend data not available: {str(e)}")

# Recent records
st.markdown("---")
st.subheader("📝 Recent Analysis Results")

try:
    recent_data = session.sql("""
        SELECT 
            r.source,
            SUBSTRING(r.content, 1, 100) as content_preview,
            a.category,
            ROUND(a.analysis_result, 3) as result,
            ROUND(a.confidence, 2) as confidence
        FROM media_articles r
        INNER JOIN media_sentiment a ON r.record_id = a.record_id
        ORDER BY a.analyzed_at DESC
        LIMIT 20
    """).to_pandas()
    
    if not recent_data.empty:
        st.dataframe(recent_data, use_container_width=True, hide_index=True)
    else:
        st.info("No analysis results yet. Run Cells 6-8 to generate data and analyze sentiment.")
except Exception as e:
    st.warning(f"Analysis results not available: {str(e)}")

st.success("✅ Dashboard operational | Data current")

---

##  Tutorial Complete!

### What You've Learned

1.  **AI-powered analysis** with Snowflake Cortex
2.  **Production data pipelines** with SQL
3.  **Aggregation and metrics** calculation
4.  **Trend analysis** with window functions
5.  **Interactive dashboards** with Streamlit

### Key Techniques

- **`SENTIMENT()`** for AI processing
- **Window functions** for trends
- **CTEs** for complex logic
- **Aggregations** for insights
- **Streamlit** for visualization

### Next Steps for Production

1. **Connect real data sources**
2. **Automate data refresh**
3. **Add alerting logic**
4. **Scale to production volumes**
5. **Integrate with reporting tools**

---

**Congratulations!** You've built a production-ready global media sentiment & share of voice tracker using Snowflake Cortex AI and SQL.

**Estimated runtime**: (varies by data size)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `GLOBAL_MEDIA_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS GLOBAL_MEDIA_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
